# SC4001 TPU probe: where does the signal-6 abort come from?

Accelerator must be **TPU**. This kernel deliberately has no CUDA check, because it expects no
CUDA: Kaggle's TPU image ships a CPU-only torch build.

## Why this kernel exists, and why it is not a retry

TPU is a recorded dead end for this project, twice over:

1. **4 Nov** — a TPU job stayed QUEUED for hours and never started, while the equivalent six-run
   GPU grid finished in 67.2 min. The conclusion was that TPU quota was never the binding
   constraint, since one DAVIS run is 8.6 to 13.2 min on a T4.
2. **5 Nov** — the DAVIS smoke on a TPU VM reported `torch 2.8.0+cpu`, `torch_xla 2.8.0`,
   resolved `torch_xla.device()` to `xla:0`, and then returned `rc -6` after 7.7 s: signal 6, a
   native abort with a core dump **inside the XLA runtime**. Cost 19 min of TPU quota.

So the blocker has never been quota. There are about 19 of 20 weekly TPU hours unused and they
cannot be spent, because runs die in under eight seconds.

This kernel does not retry that run and does not try to make TPU work. It answers one narrow
question that the previous attempt left open: **which layer aborts.** That is worth about ten
minutes of an otherwise idle resource, because the answer decides whether TPU is permanently
closed for this project or merely needs a different training loop.

## Design

Six steps, each in its **own subprocess**, because signal 6 cannot be caught in-process and
would otherwise end the probe at the first failure with nothing reported for the rest:

1. import `torch` and `torch_xla`, resolve a device
2. a trivial tensor add and sum on that device
3. `nn.Embedding(26, 128)` forward over a 1200-token input, the first layer of both encoders
4. `nn.Conv1d(128, 32, 8)` forward plus `amax(dim=2)`, which is most of what this architecture is
5. the real `DeepDTA` forward, imported from `src/model.py`
6. a backward pass and one Adam step with `torch_xla.sync()`

It stops at the first failure, since later steps build on earlier ones and their results would
be noise.

**`src/train.py` is not touched and not used.** The probe imports `src/model.py` and
`src/config.py` only, and runs its own two-step loop. Nothing here can affect an existing
result, which matters because `train.py` produced every number in the project so far and its
XLA path is verified only against an injected fake `torch_xla`.

## How to read the result

- **Fails at step 1 or 2** — the Kaggle TPU image is broken for this `torch`/`torch_xla` pair
  and nothing in this project can fix it. TPU is closed, permanently, and the finding is that
  it is an image problem rather than a model problem.
- **Fails at step 3, 4 or 5** — the abort is in a specific layer, which is a reportable fact
  about XLA and this architecture, and tells anyone extending the project which operation to
  avoid.
- **Fails at step 6** — forward works and the training step does not, so the fault is in the
  optimiser or sync path, the most fixable outcome.
- **All six pass** — the 5 Nov abort was in the training loop or the data pipeline, not the
  model, and `train.py`'s XLA path deserves a real attempt rather than being written off.

Whatever happens, the T4 stays the default route. A pass here does not make TPU the plan; it
only reopens a door that is currently recorded as shut. Note also that outcome 1 is quite
likely, and that the 4 Nov queueing problem is independent of all of this: a working stack that
never gets scheduled is still unusable.

Outputs survive only if this version is saved.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

print("python", sys.version.split()[0])
for module in ("torch", "torch_xla"):
    try:
        loaded = __import__(module)
        print(module, getattr(loaded, "__version__", "unknown"))
    except Exception as error:
        print(module, "IMPORT FAILED:", type(error).__name__, error)

# Locate the project by globbing src/train.py beside data/processed/vocab.json, never assuming
# a "Project" path segment: Kaggle strips the archive's top-level folder.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit("No project root found. Candidates with src/train.py: " + str(candidates))
project = sorted(roots)[0]
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("project:", project)


In [ ]:
# Bounded TPU probe. Reads the real DeepDTA model from src/model.py but runs its own two-step
# loop, so src/train.py is untouched and no existing result is put at risk.
#
# Background: the 5 Nov attempt resolved an XLA device fine and then aborted with signal 6
# inside the XLA runtime, 7.7 s in, with `torch 2.8.0+cpu` and `torch_xla 2.8.0`. The question
# this probe answers is WHERE that abort comes from, narrowing it one layer at a time:
#   1. import torch_xla and resolve a device
#   2. a trivial tensor op on that device
#   3. an nn.Embedding forward, the first layer of both encoders
#   4. a Conv1d forward, which is what this architecture mostly is
#   5. the full DeepDTA forward
#   6. a backward pass and one optimiser step with torch_xla.sync()
# Whichever step aborts is the answer. If all six pass, the 5 Nov abort was in the training
# loop or the data pipeline rather than the model, and TPU is worth a real attempt.
#
# Everything is wrapped so a native abort kills only the child: each step runs in its own
# subprocess, because signal 6 cannot be caught in-process and would otherwise end the probe
# at the first failure with nothing reported for the later steps.

import textwrap

STEPS = {
    "1-import-and-device": """
        import torch, torch_xla
        print("torch", torch.__version__)
        print("torch_xla", torch_xla.__version__)
        d = torch_xla.device()
        print("device", d)
    """,
    "2-trivial-op": """
        import torch, torch_xla
        d = torch_xla.device()
        a = torch.ones(8, 8, device=d)
        b = (a + a).sum()
        torch_xla.sync()
        print("sum", float(b))
    """,
    "3-embedding": """
        import torch, torch_xla
        from torch import nn
        d = torch_xla.device()
        emb = nn.Embedding(26, 128, padding_idx=0).to(d)
        x = torch.randint(1, 26, (4, 1200), device=d)
        y = emb(x)
        torch_xla.sync()
        print("embedding out", tuple(y.shape))
    """,
    "4-conv1d": """
        import torch, torch_xla
        from torch import nn
        d = torch_xla.device()
        conv = nn.Conv1d(128, 32, 8).to(d)
        x = torch.randn(4, 128, 1200, device=d)
        y = conv(x).amax(dim=2)
        torch_xla.sync()
        print("conv+amax out", tuple(y.shape))
    """,
    "5-deepdta-forward": """
        import sys, torch, torch_xla
        sys.path.insert(0, PROJECT)
        from src.config import Config
        from src.model import DeepDTA
        d = torch_xla.device()
        model = DeepDTA(30, 22, Config()).to(d)
        drugs = torch.randint(1, 30, (4, 85), device=d)
        prots = torch.randint(1, 22, (4, 1200), device=d)
        out = model(drugs, prots)
        torch_xla.sync()
        print("forward out", tuple(out.shape))
    """,
    "6-backward-and-step": """
        import sys, torch, torch_xla
        sys.path.insert(0, PROJECT)
        from src.config import Config
        from src.model import DeepDTA
        d = torch_xla.device()
        model = DeepDTA(30, 22, Config()).to(d)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        for step in range(2):
            drugs = torch.randint(1, 30, (4, 85), device=d)
            prots = torch.randint(1, 22, (4, 1200), device=d)
            target = torch.randn(4, 1, device=d)
            loss = torch.nn.functional.mse_loss(model(drugs, prots), target)
            opt.zero_grad()
            loss.backward()
            opt.step()
            torch_xla.sync()
            print("step", step, "loss", float(loss))
    """,
}

print("TPU probe: narrowing the 5 Nov signal-6 abort, one layer at a time.")
print("Each step runs in its own subprocess, because a native abort cannot be caught in-process.")
results = []
for name, body in STEPS.items():
    source = "PROJECT = " + repr(str(project)) + "\n" + textwrap.dedent(body)
    script = Path("/kaggle/working/probe_" + name.replace("-", "_") + ".py")
    script.write_text(source, encoding="utf-8")
    print()
    print("===== " + name + " =====", flush=True)
    t0 = time.time()
    run = subprocess.run([sys.executable, str(script)], cwd=project, env=env, check=False)
    elapsed = time.time() - t0
    verdict = "OK" if run.returncode == 0 else ("ABORTED (signal %d)" % -run.returncode
                                                if run.returncode < 0 else
                                                "FAILED (rc %d)" % run.returncode)
    results.append((name, run.returncode, elapsed, verdict))
    print("----- %s in %.1f s -----" % (verdict, elapsed), flush=True)
    if run.returncode != 0:
        print("Stopping here: later steps build on this one, so their results would be noise.")
        break

print()
print("=" * 78)
print("%-24s %-28s %s" % ("step", "verdict", "seconds"))
for name, code, elapsed, verdict in results:
    print("%-24s %-28s %.1f" % (name, verdict, elapsed))

passed = [row for row in results if row[1] == 0]
print()
if len(passed) == len(STEPS):
    print("ALL SIX PASSED. The 5 Nov abort was not in the model: it came from the training "
          "loop or the data pipeline. TPU is worth a real attempt, and src/train.py's XLA "
          "path should be revisited rather than written off.")
else:
    failed = results[-1]
    print("FAILED AT: " + failed[0] + " -> " + failed[3])
    print("This localises the abort. If it failed at step 1 or 2 the Kaggle TPU image itself "
          "is broken for this torch/torch_xla pair and nothing in this project can fix it. If "
          "it failed at 3, 4 or 5 the abort is in a specific layer of the architecture. Either "
          "way, record it and keep using the T4.")
